# RRG Sector Rotation Strategy - Inference Notebook

This notebook demonstrates the **Relative Rotation Graph (RRG)** sector rotation strategy.

**Sections:**
1. Configuration
2. Data Loading & Holdout Split
3. Strategy Fitting & Signal Analysis
4. Current Portfolio Weights
5. Backtest
6. Walk-Forward Validation
7. **True Out-of-Sample (Holdout) Evaluation**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from strategies import RRGStrategy, RRGConfig
from backtesting import Backtester, WalkForwardValidator, WalkForwardConfig

---
## 1. Configuration

**Modify parameters here.** Use optimized params from `optimization/` scripts or set manually.

In [ ]:
# =============================================================================
# DATA CONFIG
# =============================================================================
DATA_PATH = "data/processed/indian_sector_data_2013_2025.csv"
HOLDOUT_YEARS = 2.0  # Years to reserve for true OOS testing

# =============================================================================
# STRATEGY CONFIG (RRG)
# =============================================================================
# You can paste optimized params from optimization scripts here
config = RRGConfig(
    top_n_sectors=5,
    max_sector_weight=0.30,
    min_sector_weight=0.01,
    rs_lookback=100,
    momentum_lookback=30,
    volatility_window=46,
    use_trend_filter=True,
    trend_ma_period=50,
    weight_smoothing_alpha=0.3,
    full_allocation=True,
    rebalance_frequency="W-FRI"
)

# =============================================================================
# WALK-FORWARD CONFIG
# =============================================================================
WF_TRAIN_WINDOW = 504  # ~2 years
WF_TEST_WINDOW = 63    # ~3 months
RISK_FREE_RATE = 0.05

print("Configuration loaded.")
print(f"Holdout: last {HOLDOUT_YEARS} years reserved for true OOS")

---
## 2. Data Loading & Holdout Split

In [ ]:
# Load full dataset
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.set_index("Date").sort_index()
df = df.apply(pd.to_numeric, errors="coerce").ffill().dropna(how="all")

benchmark_full = df["Benchmark"]
prices_full = df.drop(columns=["Benchmark"])

# Split into optimization and holdout periods
holdout_days = int(HOLDOUT_YEARS * 252)
cutoff_idx = len(prices_full) - holdout_days

prices_opt = prices_full.iloc[:cutoff_idx]
benchmark_opt = benchmark_full.iloc[:cutoff_idx]

prices_holdout = prices_full.iloc[cutoff_idx:]
benchmark_holdout = benchmark_full.iloc[cutoff_idx:]

print(f"Full data:    {prices_full.index[0].date()} → {prices_full.index[-1].date()} ({len(prices_full)} days)")
print(f"Opt period:   {prices_opt.index[0].date()} → {prices_opt.index[-1].date()} ({len(prices_opt)} days)")
print(f"Holdout:      {prices_holdout.index[0].date()} → {prices_holdout.index[-1].date()} ({len(prices_holdout)} days)")
print(f"\nSectors ({len(prices_full.columns)}): {list(prices_full.columns)}")

---
## 3. Strategy Fitting & Signal Analysis

In [ ]:
# Fit on optimization period only (to avoid lookahead bias)
strategy = RRGStrategy(config)
strategy.fit(prices_opt, benchmark_opt)

print(f"Strategy fitted on optimization period")
print(config)

In [ ]:
# RRG quadrant classification
latest_opt_date = prices_opt.index[-1]
quadrants = strategy.get_quadrant_classification(latest_opt_date)

print(f"RRG Classification as of {latest_opt_date.date()}\n")
display(quadrants.sort_values("RS_Momentum", ascending=False))

In [ ]:
# Visualize RRG quadrants
fig, ax = plt.subplots(figsize=(10, 8))

colors = {
    "Leading": "green",
    "Weakening": "orange",
    "Lagging": "red",
    "Improving": "blue"
}

for sector, row in quadrants.iterrows():
    ax.scatter(row["RS_Ratio"], row["RS_Momentum"], 
               color=colors.get(row["Quadrant"], "gray"), s=100)
    ax.annotate(sector, (row["RS_Ratio"], row["RS_Momentum"]), 
                fontsize=9, ha="left")

ax.axhline(100, color="black", linewidth=0.5, linestyle="--")
ax.axvline(100, color="black", linewidth=0.5, linestyle="--")
ax.set_xlabel("RS-Ratio")
ax.set_ylabel("RS-Momentum")
ax.set_title(f"RRG Quadrant Chart - {latest_opt_date.date()}")
ax.text(101, 101, "Leading", fontsize=10, color="green")
ax.text(101, 99, "Weakening", fontsize=10, color="orange")
ax.text(99, 99, "Lagging", fontsize=10, color="red")
ax.text(99, 101, "Improving", fontsize=10, color="blue")
plt.tight_layout()
plt.show()

---
## 4. Current Portfolio Weights

In [ ]:
# Portfolio weights
weights = strategy.predict_weights(prices_opt, latest_opt_date)

print(f"Portfolio Weights as of {latest_opt_date.date()}")
print(f"Total Allocation: {weights.sum():.2%}\n")

active = weights[weights > 0].sort_values(ascending=False)
for sector, w in active.items():
    print(f"  {sector}: {w:.2%}")

---
## 5. Backtest (Full Period)

In [ ]:
# Backtest on full data
strategy_full = RRGStrategy(config)
backtester = Backtester(strategy_full, risk_free_rate=RISK_FREE_RATE)
result = backtester.run(prices_full, benchmark_full)

print("=" * 60)
print("RRG STRATEGY BACKTEST (Full Period)")
print("=" * 60)
backtester.print_report(result)

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(result.strategy_equity, label="Strategy", linewidth=2)
ax.plot(result.benchmark_equity, label="Benchmark", linewidth=1, alpha=0.7)
ax.axvline(prices_holdout.index[0], color="green", linestyle=":", linewidth=2, label="Holdout Start")

ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Strategy vs Benchmark (Full Period)")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Plot drawdowns
def compute_drawdown(equity):
    return (equity / equity.cummax()) - 1

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(result.strategy_equity.index, 
                compute_drawdown(result.strategy_equity), 
                0, alpha=0.5, label="Strategy DD")
ax.fill_between(result.benchmark_equity.index, 
                compute_drawdown(result.benchmark_equity), 
                0, alpha=0.3, label="Benchmark DD")
ax.axvline(prices_holdout.index[0], color="green", linestyle=":", linewidth=2)
ax.set_ylabel("Drawdown")
ax.set_title("Drawdown Comparison")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Walk-Forward Validation (Optimization Period Only)

In [ ]:
# Walk-forward on optimization period (not holdout)
wf_config = WalkForwardConfig(
    train_window=WF_TRAIN_WINDOW,
    test_window=WF_TEST_WINDOW,
    step_size=WF_TEST_WINDOW
)

strategy_wf = RRGStrategy(config)
validator = WalkForwardValidator(strategy_wf, config=wf_config, risk_free_rate=RISK_FREE_RATE)
wf_result = validator.validate(prices_opt, benchmark_opt)

print("=" * 60)
print("WALK-FORWARD (Optimization Period Only)")
print("=" * 60)
validator.print_report(wf_result)

In [ ]:
# Plot OOS equity vs equal-weight benchmark (both starting at 1.0)
fig, ax = plt.subplots(figsize=(12, 6))

# Get OOS returns and build equity from returns
oos_start = wf_result.oos_returns.index[0]
oos_end = wf_result.oos_returns.index[-1]

# Strategy equity from returns (starts at 1.0)
oos_equity = (1 + wf_result.oos_returns).cumprod()

# Equal-weight benchmark from sector returns (starts at 1.0)
prices_oos = prices_opt.loc[oos_start:oos_end]
sector_returns = prices_oos.pct_change().fillna(0)
mean_returns = sector_returns.mean(axis=1)
benchmark_oos = (1 + mean_returns).cumprod()

ax.plot(oos_equity, label="Strategy (OOS)", linewidth=2)
ax.plot(benchmark_oos, label="Equal-Weight Benchmark", linewidth=1.5, alpha=0.7, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Equity (Starting at 1.0)")
ax.set_title("Walk-Forward OOS vs Equal-Weight Benchmark (Opt Period)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-fold performance
wf_result.fold_metrics[["fold", "test_start", "test_end", "total_return", "max_drawdown"]]

---
## 7. True Out-of-Sample (Holdout) Evaluation

**CRITICAL**: This section shows performance on data that was NEVER used during:
- Parameter optimization
- Walk-forward validation

This is the most reliable indicator of future performance.

In [ ]:
# Strategy was already fitted on opt period - now test on holdout
backtester_holdout = Backtester(strategy, risk_free_rate=RISK_FREE_RATE)
holdout_result = backtester_holdout.run(prices_holdout, benchmark_holdout)

print("=" * 70)
print("TRUE OUT-OF-SAMPLE EVALUATION (Holdout Period)")
print("=" * 70)
print(f"Period: {prices_holdout.index[0].date()} → {prices_holdout.index[-1].date()}")
print("Note: Strategy was fitted ONLY on optimization period data!\n")
backtester_holdout.print_report(holdout_result)

In [ ]:
# Plot holdout equity vs equal-weight benchmark (both starting at 1.0)
fig, ax = plt.subplots(figsize=(12, 6))

# Strategy equity (normalize to start at 1.0)
strategy_equity = holdout_result.strategy_equity / holdout_result.strategy_equity.iloc[0]

# Equal-weight benchmark for holdout (starts at 1.0)
holdout_returns = prices_holdout.pct_change().fillna(0)
holdout_mean_returns = holdout_returns.mean(axis=1)
holdout_benchmark = (1 + holdout_mean_returns).cumprod()

ax.plot(strategy_equity, label="Strategy", linewidth=2)
ax.plot(holdout_benchmark, label="Equal-Weight Benchmark", linewidth=1.5, alpha=0.7, linestyle="--")

ax.set_xlabel("Date")
ax.set_ylabel("Equity (Starting at 1.0)")
ax.set_title(f"TRUE OOS Performance ({prices_holdout.index[0].date()} → {prices_holdout.index[-1].date()})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison
print("\n" + "=" * 70)
print("PERFORMANCE SUMMARY")
print("=" * 70)
print(f"{'Metric':<20} {'Opt WF OOS':<15} {'True Holdout':<15}")
print("-" * 50)

wf_metrics = wf_result.aggregate_metrics
ho_metrics = holdout_result.metrics

metrics_to_show = [
    ("Total Return", "total_return", ".1%"),
    ("CAGR", "cagr", ".1%"),
    ("Sharpe Ratio", "sharpe_ratio", ".3f"),
    ("Max Drawdown", "max_drawdown", ".1%"),
]

for label, key, fmt in metrics_to_show:
    wf_val = wf_metrics.get(key, float('nan'))
    ho_val = ho_metrics.get(key, float('nan'))
    print(f"{label:<20} {wf_val:{fmt}:<15} {ho_val:{fmt}:<15}")

---

## Summary

The **RRG Strategy** uses:

1. **RS-Ratio** - Relative strength vs benchmark
2. **RS-Momentum** - Rate of change in relative strength
3. **Quadrant Classification** - Leading, Weakening, Lagging, Improving
4. **Holdout Validation** - True OOS test on unseen data

Compare **Opt WF OOS** vs **True Holdout** metrics above to assess overfitting risk.